In [ ]:
# Joining the flights with the weather data

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = "/home/jovyan/project"
CLEAN = ROOT + "/data/clean"
OUT = ROOT + "/data/joined"

spark = (SparkSession.builder.appName("dpa-join").config("spark.driver.memory", "2g").config("spark.sql.session.timeZone", "UTC").getOrCreate())

flights = spark.read.parquet(CLEAN + "/flights")
airports = spark.read.parquet(CLEAN + "/airports")
weather = spark.read.parquet(CLEAN + "/weather")
stations = spark.read.parquet(ROOT + "/data/parquet/stations")   # made by ingest.py: iata, icao, station

counts = []
n = flights.count()
counts.append(("clean flights", n))
print("clean flights:", n)

In [ ]:
# airpost data does not provide the timezone so we are usign the timzone of the state

In [ ]:
eastern = ["CT", "DE", "DC", "FL", "GA", "IN", "KY", "ME", "MD", "MA", "MI", "NH", "NJ", "NY",
           "NC", "OH", "PA", "RI", "SC", "VT", "VA", "WV"]
central = ["AL", "AR", "IL", "IA", "KS", "LA", "MN", "MS", "MO", "NE", "ND", "OK", "SD", "TN", "TX", "WI"]
mountain = ["CO", "ID", "MT", "NM", "UT", "WY"]
pacific = ["CA", "NV", "OR", "WA"]

origin = airports.select(F.col("iata_code").alias("Origin"),
                         F.col("name").alias("origin_name"),
                         F.regexp_replace("iso_region", "US-", "").alias("state"))

state = F.col("state")
origin = origin.withColumn("timezone",
                           F.when(state.isin(eastern), "America/New_York")
                            .when(state.isin(central), "America/Chicago")
                            .when(state.isin(mountain), "America/Denver")
                            .when(state.isin(pacific), "America/Los_Angeles")
                            .when(state == "AZ", "America/Phoenix")
                            .when(state == "HI", "Pacific/Honolulu")
                            .when(state == "AK", "America/Anchorage"))
origin = origin.filter(F.col("timezone").isNotNull())

In [ ]:
# building the local departure hour and convret it the UTC

In [ ]:
flights = flights.join(origin, "Origin")
n = flights.count()
counts.append(("flights with a known origin airport", n))
print("flights with a known origin airport:", n)

flights = flights.withColumn("dep_hour_local",
                             F.to_timestamp(F.concat(F.col("FlightDate"), F.lit(" "), F.lpad("dep_hour", 2, "0")),
                                            "yyyy-MM-dd HH"))
flights = flights.withColumn("dep_hour_utc", F.to_utc_timestamp("dep_hour_local", F.col("timezone")))
flights.select("FlightDate", "Origin", "CRSDepTime", "dep_hour_local", "timezone", "dep_hour_utc").show(5, truncate=False)